In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:

# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    ano_usado_mais_um = int(ano_usado)+1
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0.0).all()])
    dfs[ano_usado] = dfs[ano_usado].loc[str(ano_usado_mais_um)]
    ativos_manter = (dfs[ano_usado]==0.0).mean()
    c_manter = ativos_manter[ativos_manter<0.5].index
    dfs[ano_usado] = dfs[ano_usado][c_manter]

    print(ano,'-',len(dfs[ano_usado].columns))

2025-12-31 - 100
2024-12-31 - 99
2023-12-31 - 99
2022-12-31 - 99
2021-12-31 - 97
2020-12-31 - 84
2019-12-31 - 77
2018-12-31 - 76
2017-12-31 - 73
2016-12-31 - 69
2015-12-31 - 68


## Hiperparâmetros

In [4]:
vb_cardinalidade_max = 100
# vb_cardinalidade_min = 10
vb_peso_maximo = 0.15
# vb_peso_minimo = 0.02
vb_theta = 0.25

### Fazendo otimização ano a ano e comparando com o proximo ano

In [5]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [6]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    try:
        ano_um = int(ano)
        # df_usado = f'df_ativos_{ano_um}'
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)]
        retorno_usado = df_usado.copy()

        score_todos_anos = pd.read_excel(f'../../../score-raspagem/piotroski/resultado/{ano_um}.xlsx')
        score_usado_p = score_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
        
        ativos = retorno_usado.columns.intersection(score_usado_p.columns)
        retorno_usado = retorno_usado[ativos]
        retorno_usado = (retorno_usado - retorno_usado.min())/(retorno_usado.max() - retorno_usado.min())

        score = score_usado_p[ativos]

        print("Retornos atualizados")

        sigma_usado = retorno_usado.cov()
        print("Sigmas Atualizados")
        
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns.tolist())
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns.tolist())-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    # model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    # model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos,bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        min_var = sum(model.x[a]*model.x[b]*model.sigma[a,b] for a in model.ativos for b in model.ativos)
        score = sum(model.x[a] * model.score[a] for a in model.ativos)
        return model.theta*min_var - (1-model.theta)*score
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)
    #=============================
    # RESTRIÇÕES
    #=============================
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo * model.y[a]  # se y=1, então x <= 0.20
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)

    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    # opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)

    melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Retornos atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO DADOS DE RETORNO DE:  2015
## UTILIZANDO SIGMAS DO ANO DE:  2015
2015 -> {'ABEV3': 0.0, 'ALOS3': 0.0, 'ALPA4': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.09999999999999981, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.0, 'ENEV3': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.15000000000000002, 'GGBR4': 0.0, 'GOAU4': 0.0, 'HYPE3': 0.0, 'ISAE4': 0.15000000000000002, 'ITSA4': 0.0, 'JHSF3':

In [7]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.0,
   'ALOS3': 0.0,
   'ALPA4': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BEEF3': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.09999999999999981,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CYRE3': 0.0,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EGIE3': 0.0,
   'EMBJ3': 0.0,
   'ENEV3': 0.0,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.15000000000000002,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.15000000000000002,
   'ITSA4': 0.0,
   'JHSF3': 0.0,
   'KLBN11': 0.0,
   'LREN3': 0.0,
   'MBRF3': 0.0,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MRVE3': 0.15000000000000002,
   'MULT3': 0.0,
   'NATU3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.0,
   'PRIO3': 0.0,
   'RADL3': 0.15000000000000002,
   'RAIL3': 0.0,
   'RENT3': 0.0,
   'SBSP3': 0.0,
   'SLCE3': 0.15000000000000002,
   'SMTO3': 0.0,
   'TAEE11': 0.0,
   'TOTS3': 0.0,
   'UGPA3': 0.15000000000000002,
   '

In [8]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [18]:
df_portfolios[df_portfolios['ano'] == 2026].sort_values('peso', ascending=False)['peso']

87    0.150000
89    0.150000
90    0.150000
92    0.150000
93    0.150000
94    0.150000
88    0.072625
91    0.027375
Name: peso, dtype: float64

In [10]:
df_portfolios.to_csv('../carteiras/min_var_piotroski/carteiras_minvar_piotroski.csv')
